# LPN Encoder-Only Training

このノートブックでは、LPN（Latent Program Network）のエンコーダーのみを学習します。
デコーダーのパラメータを固定し、エンコーダーの潜在表現学習に集中します。

## 目的
- エンコーダーの潜在表現能力を評価
- 変分推論（VAE）の効果を確認
- 潜在空間の可視化と分析
- デコーダーなしでの表現学習の検証

## 1. 環境設定とインポート

In [3]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Dict, Tuple
import time

# JAX関連
import jax
import jax.numpy as jnp
from jax import random, grad, vmap
from jax.tree_util import tree_map
import optax
from flax import linen as nn
from flax.training.train_state import TrainState
import chex

# 可視化
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go

# プロジェクト固有
from src.models.lpn import LPN
from src.models.transformer import EncoderTransformer, DecoderTransformer
from src.models.utils import EncoderTransformerConfig, DecoderTransformerConfig, TransformerLayerConfig
from src.data_utils import make_leave_one_out, data_augmentation_fn
from src.datasets.task_gen.dataloader import make_task_gen_dataloader

print(f"JAX devices: {jax.devices()}")
print(f"JAX backend: {jax.default_backend()}")

ModuleNotFoundError: No module named 'jax'

In [ ]:
pip install jax

## 2. エンコーダー学習専用クラス

In [ ]:
class EncoderOnlyTrainer:
    """エンコーダーのみを学習するためのトレーナークラス"""
    
    def __init__(self, encoder_config: EncoderTransformerConfig, 
                 decoder_config: DecoderTransformerConfig,
                 learning_rate: float = 1e-3,
                 prior_kl_coeff: float = 1e-4,
                 pairwise_kl_coeff: float = 1e-5):
        
        # モデル構築
        self.encoder = EncoderTransformer(encoder_config)
        self.decoder = DecoderTransformer(decoder_config)
        self.lpn = LPN(encoder=self.encoder, decoder=self.decoder)
        
        # 学習パラメータ
        self.learning_rate = learning_rate
        self.prior_kl_coeff = prior_kl_coeff
        self.pairwise_kl_coeff = pairwise_kl_coeff
        
        # オプティマイザー
        self.optimizer = optax.adam(learning_rate=learning_rate)
        
        # メトリクス保存
        self.train_metrics_history = []
        
    def init_train_state(self, key: chex.PRNGKey, sample_data: Tuple) -> TrainState:
        """学習状態を初期化"""
        pairs, grid_shapes = sample_data
        
        # モデル初期化
        variables = self.lpn.init(
            key, pairs, grid_shapes, 
            dropout_eval=False, 
            mode="mean",
            prior_kl_coeff=self.prior_kl_coeff,
            pairwise_kl_coeff=self.pairwise_kl_coeff
        )
        
        # TrainState作成
        state = TrainState.create(
            apply_fn=self.lpn.apply,
            params=variables["params"],
            tx=self.optimizer
        )
        
        return state
    
    def encoder_only_loss_fn(self, params: dict, pairs: chex.Array, 
                            grid_shapes: chex.Array, key: chex.PRNGKey) -> Tuple[float, dict]:
        """エンコーダーのみの損失関数"""
        
        # エンコーダーのみ実行
        latents_mu, latents_logvar = self.encoder.apply(
            {"params": params["encoder"]}, 
            pairs, grid_shapes, dropout_eval=False
        )
        
        if latents_logvar is not None:
            # 変分推論の場合
            latents, prior_kl_loss, kl_metrics = self.lpn._sample_latents(
                latents_mu, latents_logvar, key
            )
            
            # ペアワイズKL計算
            pairwise_kl_loss = self.lpn._compute_pairwise_gaussian_kl(
                latents_mu, latents_logvar
            ).mean()
            
            # 総損失
            loss = (self.prior_kl_coeff * prior_kl_loss + 
                   self.pairwise_kl_coeff * pairwise_kl_loss)
            
            metrics = {
                "prior_kl": prior_kl_loss,
                "pairwise_kl": pairwise_kl_loss,
                "total_loss": loss,
                "latents_norm": jnp.mean(jnp.linalg.norm(latents, axis=-1)),
                "latents_mu_norm": jnp.mean(jnp.linalg.norm(latents_mu, axis=-1)),
                "latents_logvar_mean": jnp.mean(latents_logvar)
            }
            metrics.update(kl_metrics)
            
        else:
            # 決定的な場合
            latents = latents_mu
            
            # 正則化損失のみ
            loss = jnp.mean(jnp.sum(latents**2, axis=-1))
            
            metrics = {
                "regularization_loss": loss,
                "total_loss": loss,
                "latents_norm": jnp.mean(jnp.linalg.norm(latents, axis=-1))
            }
        
        return loss, metrics
    
    def train_step(self, state: TrainState, batch: Tuple, key: chex.PRNGKey) -> Tuple[TrainState, dict]:
        """エンコーダーのみの学習ステップ"""
        pairs, grid_shapes = batch
        
        # 勾配計算（エンコーダーのみ）
        grads, metrics = grad(self.encoder_only_loss_fn, has_aux=True)(
            state.params, pairs, grid_shapes, key
        )
        
        # デコーダーの勾配をゼロに設定
        grads = grads.copy({
            "decoder": jax.tree_map(jnp.zeros_like, grads["decoder"])
        })
        
        # パラメータ更新
        state = state.apply_gradients(grads=grads)
        
        # 勾配ノルム追加
        metrics["grad_norm"] = optax.global_norm(grads)
        
        return state, metrics
    
    def extract_latents(self, state: TrainState, pairs: chex.Array, 
                       grid_shapes: chex.Array) -> Tuple[chex.Array, chex.Array]:
        """学習済みエンコーダーから潜在変数を抽出"""
        latents_mu, latents_logvar = self.encoder.apply(
            {"params": state.params["encoder"]},
            pairs, grid_shapes, dropout_eval=True
        )
        return latents_mu, latents_logvar
    
    def train(self, dataloader, num_epochs: int = 10, 
             print_every: int = 100) -> TrainState:
        """学習実行"""
        # サンプルデータで初期化
        sample_batch = next(iter(dataloader))
        key = random.PRNGKey(42)
        state = self.init_train_state(key, sample_batch)
        
        print(f"開始: エンコーダーのみ学習 ({num_epochs} epochs)")
        print(f"パラメータ数: {sum(p.size for p in jax.tree_leaves(state.params)):,}")
        
        step = 0
        start_time = time.time()
        
        for epoch in range(num_epochs):
            epoch_metrics = []
            
            for batch in dataloader:
                key, step_key = random.split(key)
                state, metrics = self.train_step(state, batch, step_key)
                epoch_metrics.append(metrics)
                
                step += 1
                
                if step % print_every == 0:
                    elapsed = time.time() - start_time
                    avg_loss = jnp.mean(jnp.array([m["total_loss"] for m in epoch_metrics[-print_every:]]))
                    print(f"Step {step:6d} | Loss: {avg_loss:.6f} | Time: {elapsed:.1f}s")
            
            # エポック終了時のメトリクス
            epoch_avg_metrics = tree_map(
                lambda *args: jnp.mean(jnp.array(args)), 
                *epoch_metrics
            )
            self.train_metrics_history.append(epoch_avg_metrics)
            
            print(f"Epoch {epoch+1:3d} | Avg Loss: {epoch_avg_metrics['total_loss']:.6f}")
        
        print(f"学習完了！総時間: {time.time() - start_time:.1f}s")
        return state

## 3. 設定とデータ準備

In [ ]:
# モデル設定
vocab_size = 10
max_rows = 5
max_cols = 5
latent_dim = 64
emb_dim = 128

# エンコーダー設定
encoder_config = EncoderTransformerConfig(
    vocab_size=vocab_size,
    max_rows=max_rows,
    max_cols=max_cols,
    emb_dim=emb_dim,
    latent_dim=latent_dim,
    num_layers=4,
    transformer_layer=TransformerLayerConfig(
        num_heads=8,
        mlp_dim=256,
        dropout_rate=0.1
    ),
    variational=True,  # 変分推論を有効化
    scaled_position_embeddings=True
)

# デコーダー設定（固定用）
decoder_config = DecoderTransformerConfig(
    vocab_size=vocab_size,
    max_rows=max_rows,
    max_cols=max_cols,
    emb_dim=emb_dim,
    num_layers=4,
    transformer_layer=TransformerLayerConfig(
        num_heads=8,
        mlp_dim=256,
        dropout_rate=0.1
    )
)

# 学習設定
learning_rate = 1e-3
prior_kl_coeff = 1e-4
pairwise_kl_coeff = 1e-5
batch_size = 32
num_epochs = 20

print("設定完了:")
print(f"  - 語彙サイズ: {vocab_size}")
print(f"  - 最大グリッドサイズ: {max_rows}x{max_cols}")
print(f"  - 潜在次元: {latent_dim}")
print(f"  - 埋め込み次元: {emb_dim}")
print(f"  - 変分推論: {encoder_config.variational}")
print(f"  - 学習率: {learning_rate}")
print(f"  - バッチサイズ: {batch_size}")

## 4. データ準備 (ARC データセット)

以前の合成データ生成の代わりに、`data_utils` を使用して実際のARCデータセットをロードします。
これにより、エンコーダーはより現実的なデータで学習できます。

In [ ]:
# データセットのロード
# 注意: use_hf=True にすると、Hugging Face Hubからデータセットをダウンロードします。
# ローカルにデータがある場合は use_hf=False に設定してください。
try:
    train_datasets = load_datasets(["arc_train/"], use_hf=True)
    val_datasets = load_datasets(["arc_val/"], use_hf=True)

    # 複数のデータセットを結合
    train_grids = jnp.concatenate([d[0] for d in train_datasets], axis=0)
    train_shapes = jnp.concatenate([d[1] for d in train_datasets], axis=0)
    train_program_ids = jnp.concatenate([d[2] for d in train_datasets], axis=0)

    val_grids = jnp.concatenate([d[0] for d in val_datasets], axis=0)
    val_shapes = jnp.concatenate([d[1] for d in val_datasets], axis=0)
    val_program_ids = jnp.concatenate([d[2] for d in val_datasets], axis=0)

    print(f"訓練データ: {train_grids.shape}, プログラムID: {train_program_ids.shape}")
    print(f"検証データ: {val_grids.shape}, プログラムID: {val_program_ids.shape}")

except Exception as e:
    print(f"データセットのロードに失敗しました: {e}")
    print("代わりに合成データセットを使用します。")
    # フォールバックとして合成データを作成
    def create_synthetic_dataset(num_samples: int = 1000, num_pairs: int = 3) -> Tuple[chex.Array, chex.Array, chex.Array]:
        key = random.PRNGKey(42)
        pairs = random.randint(key, (num_samples, num_pairs, max_rows, max_cols, 2), 0, vocab_size)
        shapes = random.randint(key, (num_samples, num_pairs, 2, 2), 1, min(max_rows, max_cols) + 1)
        program_ids = random.randint(key, (num_samples,), 0, 50) # 合成プログラムID
        return pairs, shapes, program_ids

    train_grids, train_shapes, train_program_ids = create_synthetic_dataset(num_samples=2000, num_pairs=3)
    val_grids, val_shapes, val_program_ids = create_synthetic_dataset(num_samples=500, num_pairs=3)

# データローダー作成
def create_dataloader(grids: chex.Array, shapes: chex.Array, program_ids: chex.Array,
                     batch_size: int, shuffle: bool = True):
    num_samples = grids.shape[0]
    indices = np.arange(num_samples)

    if shuffle:
        np.random.shuffle(indices)

    for i in range(0, num_samples, batch_size):
        batch_indices = indices[i:i+batch_size]
        yield grids[batch_indices], shapes[batch_indices], program_ids[batch_indices]

train_dataloader = list(create_dataloader(train_grids, train_shapes, train_program_ids, batch_size))
val_dataloader = list(create_dataloader(val_grids, val_shapes, val_program_ids, batch_size, shuffle=False))

print(f"\n訓練バッチ数: {len(train_dataloader)}")
print(f"検証バッチ数: {len(val_dataloader)}")

### データ可視化

`visualization.py` のユーティリティを使って、ロードしたデータがどのようなものか確認します。

In [ ]:
from src.visualization import display_function_examples

print("訓練データからランダムなタスク例を表示:")
# display_function_examples は内部でランダムにバッチを選ぶため、
# データセット全体を渡します。
display_function_examples(train_grids, train_shapes, num_pairs=3)

### データ拡張

`data_utils.data_augmentation_fn` を使って、回転や色の置換といったデータ拡張を適用できます。
これにより、モデルの汎化性能が向上します。

In [ ]:
# データ拡張の適用例
key = random.PRNGKey(101)
sample_grids, sample_shapes, _ = next(iter(train_dataloader))

print("拡張前のデータ:")
display_function_examples(sample_grids, sample_shapes, num_pairs=3)

augmented_grids, augmented_shapes = data_augmentation_fn(sample_grids, sample_shapes, key)

print("\n拡張後のデータ:")
display_function_examples(augmented_grids, augmented_shapes, num_pairs=3)

# トレーニングステップ内でこれを適用することができます
# (このノートブックでは簡略化のため省略しますが、実際の学習では重要です)

## 5. エンコーダー学習実行

In [ ]:
# トレーナー初期化
trainer = EncoderOnlyTrainer(
    encoder_config=encoder_config,
    decoder_config=decoder_config,
    learning_rate=learning_rate,
    prior_kl_coeff=prior_kl_coeff,
    pairwise_kl_coeff=pairwise_kl_coeff
)

# 学習実行
print("🚀 エンコーダーのみ学習開始")
print("=" * 50)

trained_state = trainer.train(
    dataloader=train_dataloader,
    num_epochs=num_epochs,
    print_every=50
)

print("\n✅ 学習完了!")

## 6. 学習結果の可視化

In [ ]:
# 学習曲線のプロット
def plot_training_curves(metrics_history):
    """学習曲線を可視化"""
    epochs = range(1, len(metrics_history) + 1)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('エンコーダー学習曲線', fontsize=16)
    
    # 総損失
    axes[0, 0].plot(epochs, [m['total_loss'] for m in metrics_history], 'b-', linewidth=2)
    axes[0, 0].set_title('総損失')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].grid(True)
    
    # 事前分布KL
    axes[0, 1].plot(epochs, [m['prior_kl'] for m in metrics_history], 'r-', linewidth=2)
    axes[0, 1].set_title('事前分布KL')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('KL Divergence')
    axes[0, 1].grid(True)
    
    # ペアワイズKL
    axes[0, 2].plot(epochs, [m['pairwise_kl'] for m in metrics_history], 'g-', linewidth=2)
    axes[0, 2].set_title('ペアワイズKL')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('KL Divergence')
    axes[0, 2].grid(True)
    
    # 潜在変数ノルム
    axes[1, 0].plot(epochs, [m['latents_norm'] for m in metrics_history], 'm-', linewidth=2)
    axes[1, 0].set_title('潜在変数ノルム')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Norm')
    axes[1, 0].grid(True)
    
    # 勾配ノルム
    axes[1, 1].plot(epochs, [m['grad_norm'] for m in metrics_history], 'c-', linewidth=2)
    axes[1, 1].set_title('勾配ノルム')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Gradient Norm')
    axes[1, 1].grid(True)
    
    # 潜在変数平均のノルム
    axes[1, 2].plot(epochs, [m['latents_mu_norm'] for m in metrics_history], 'orange', linewidth=2)
    axes[1, 2].set_title('潜在変数平均ノルム')
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].set_ylabel('Mean Norm')
    axes[1, 2].grid(True)
    
    plt.tight_layout()
    plt.show()

# 学習曲線可視化
plot_training_curves(trainer.train_metrics_history)

## 7. 潜在空間の分析

In [ ]:
# 検証データから潜在変数を抽出
def extract_all_latents(state, dataloader):
    """全データから潜在変数を抽出"""
    all_latents_mu = []
    all_latents_logvar = []
    all_program_ids = []
    
    for pairs, grid_shapes, program_ids in dataloader:
        latents_mu, latents_logvar = trainer.extract_latents(state, pairs, grid_shapes)
        all_latents_mu.append(latents_mu)
        all_program_ids.append(program_ids)
        if latents_logvar is not None:
            all_latents_logvar.append(latents_logvar)
    
    all_latents_mu = jnp.concatenate(all_latents_mu, axis=0)
    all_program_ids = jnp.concatenate(all_program_ids, axis=0)
    if all_latents_logvar:
        all_latents_logvar = jnp.concatenate(all_latents_logvar, axis=0)
    else:
        all_latents_logvar = None
    
    return all_latents_mu, all_latents_logvar, all_program_ids

print("潜在変数抽出中...")
val_latents_mu, val_latents_logvar, val_program_ids_extracted = extract_all_latents(trained_state, val_dataloader)

print(f"抽出された潜在変数形状: {val_latents_mu.shape}")
if val_latents_logvar is not None:
    print(f"対数分散形状: {val_latents_logvar.shape}")

# 潜在変数の統計
print("\n潜在変数統計:")
print(f"平均ノルム: {jnp.mean(jnp.linalg.norm(val_latents_mu, axis=-1)):.4f}")
print(f"標準偏差: {jnp.std(val_latents_mu):.4f}")
if val_latents_logvar is not None:
    print(f"平均対数分散: {jnp.mean(val_latents_logvar):.4f}")

## 8. 潜在空間の可視化 (t-SNE)

`visualization.visualize_tsne` を使用して、潜在空間を可視化します。
この関数はプログラムIDに基づいて各点に色を付け、凡例を表示するため、より詳細な分析が可能です。

In [ ]:
from src.visualization import visualize_tsne

# t-SNE可視化実行
# タスク（プログラム）ごとに潜在表現がクラスタリングされているか確認します
print("t-SNE可視化実行中...")

# 潜在変数は (num_tasks, num_pairs, latent_dim) の形状を持つため、
# タスクごとに平均化して (num_tasks, latent_dim) にします
latents_for_tsne = val_latents_mu.mean(axis=1)

fig = visualize_tsne(
    latents=latents_for_tsne,
    program_ids=val_program_ids_extracted,
    perplexity=min(30.0, len(latents_for_tsne) - 1.0), # Perplexityはサンプル数-1より小さくする必要がある
    max_iter=1000,
    random_state=42
)

if fig:
    fig.show()
else:
    print("t-SNEの可視化に失敗しました。データ点を確認してください。")

In [ ]:
# PCA可視化
def visualize_latent_space_pca(latents_mu, program_ids, n_samples=1000):
    """PCAによる潜在空間可視化（プログラムIDで色分け）"""
    # タスクごとに平均化
    latents_agg = latents_mu.mean(axis=1)
    
    # サンプリング
    if len(latents_agg) > n_samples:
        indices = np.random.choice(len(latents_agg), n_samples, replace=False)
        latents_sample = latents_agg[indices]
        program_ids_sample = program_ids[indices]
    else:
        latents_sample = latents_agg
        program_ids_sample = program_ids

    # PCA実行
    pca = PCA(n_components=3)
    latents_pca = pca.fit_transform(np.array(latents_sample))
    
    # 3D散布図 (Plotly)
    fig = px.scatter_3d(
        x=latents_pca[:, 0],
        y=latents_pca[:, 1],
        z=latents_pca[:, 2],
        color=program_ids_sample.astype(str),
        title='潜在空間のPCA可視化 (プログラムID別)',
        labels={'x': f'PC1 ({pca.explained_variance_ratio_[0]:.2%})', 
                'y': f'PC2 ({pca.explained_variance_ratio_[1]:.2%})', 
                'z': f'PC3 ({pca.explained_variance_ratio_[2]:.2%})'},
        opacity=0.8
    )
    fig.update_traces(marker=dict(size=4))
    fig.show()
    
    print(f"寄与率合計 (PC1-3): {sum(pca.explained_variance_ratio_[:3]):.2%}")
    return pca

# PCA可視化実行
pca_model = visualize_latent_space_pca(val_latents_mu, val_program_ids_extracted)

## 9. 潜在変数の分布分析

In [ ]:
# 潜在変数の分布分析
def analyze_latent_distributions(latents_mu, latents_logvar=None):
    """潜在変数の分布を分析"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('潜在変数分布分析', fontsize=16)
    
    # 潜在変数の平均分布
    latents_flat = latents_mu.reshape(-1, latents_mu.shape[-1])
    
    # 各次元の平均
    axes[0, 0].hist(np.mean(latents_flat, axis=0), bins=30, alpha=0.7, color='blue')
    axes[0, 0].set_title('各次元の平均値分布')
    axes[0, 0].set_xlabel('平均値')
    axes[0, 0].set_ylabel('頻度')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 各次元の標準偏差
    axes[0, 1].hist(np.std(latents_flat, axis=0), bins=30, alpha=0.7, color='red')
    axes[0, 1].set_title('各次元の標準偏差分布')
    axes[0, 1].set_xlabel('標準偏差')
    axes[0, 1].set_ylabel('頻度')
    axes[0, 1].grid(True, alpha=0.3)
    
    # ノルム分布
    norms = np.linalg.norm(latents_flat, axis=1)
    axes[0, 2].hist(norms, bins=50, alpha=0.7, color='green')
    axes[0, 2].set_title('潜在変数ノルム分布')
    axes[0, 2].set_xlabel('ノルム')
    axes[0, 2].set_ylabel('頻度')
    axes[0, 2].grid(True, alpha=0.3)
    
    # 相関行列のヒートマップ
    # サンプリング（大きすぎる場合）
    if len(latents_flat) > 1000:
        sample_indices = np.random.choice(len(latents_flat), 1000, replace=False)
        latents_sample = latents_flat[sample_indices]
    else:
        latents_sample = latents_flat
    
    corr_matrix = np.corrcoef(latents_sample.T)
    im = axes[1, 0].imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
    axes[1, 0].set_title('潜在次元間相関')
    axes[1, 0].set_xlabel('次元')
    axes[1, 0].set_ylabel('次元')
    plt.colorbar(im, ax=axes[1, 0])
    
    # 最初の数次元の値分布
    for i in range(min(5, latents_flat.shape[1])):
        axes[1, 1].hist(latents_flat[:, i], bins=30, alpha=0.5, label=f'次元{i}')
    axes[1, 1].set_title('主要次元の値分布')
    axes[1, 1].set_xlabel('値')
    axes[1, 1].set_ylabel('頻度')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # 変分推論の場合：分散の分析
    if latents_logvar is not None:
        logvar_flat = latents_logvar.reshape(-1, latents_logvar.shape[-1])
        var_flat = np.exp(logvar_flat)
        
        axes[1, 2].hist(np.mean(var_flat, axis=0), bins=30, alpha=0.7, color='purple')
        axes[1, 2].set_title('各次元の分散分布')
        axes[1, 2].set_xlabel('分散')
        axes[1, 2].set_ylabel('頻度')
        axes[1, 2].grid(True, alpha=0.3)
    else:
        axes[1, 2].text(0.5, 0.5, '変分推論なし', ha='center', va='center', transform=axes[1, 2].transAxes)
        axes[1, 2].set_title('分散情報')
    
    plt.tight_layout()
    plt.show()
    
    # 統計情報出力
    print("潜在変数統計:")
    print(f"  平均ノルム: {np.mean(norms):.4f} ± {np.std(norms):.4f}")
    print(f"  平均値の範囲: [{np.min(np.mean(latents_flat, axis=0)):.4f}, {np.max(np.mean(latents_flat, axis=0)):.4f}]")
    print(f"  標準偏差の範囲: [{np.min(np.std(latents_flat, axis=0)):.4f}, {np.max(np.std(latents_flat, axis=0)):.4f}]")
    
    if latents_logvar is not None:
        logvar_flat = latents_logvar.reshape(-1, latents_logvar.shape[-1])
        print(f"  対数分散の範囲: [{np.min(logvar_flat):.4f}, {np.max(logvar_flat):.4f}]")

# 分布分析実行
analyze_latent_distributions(val_latents_mu, val_latents_logvar)

## 10. エンコーダー性能評価

In [ ]:
# エンコーダーの再構成能力テスト（潜在変数の品質確認）
def evaluate_encoder_quality(state, test_dataloader):
    """エンコーダーの品質を評価"""
    total_prior_kl = 0
    total_pairwise_kl = 0
    total_samples = 0
    latent_norms = []
    
    print("エンコーダー品質評価中...")
    
    for pairs, grid_shapes in test_dataloader:
        # 潜在変数抽出
        latents_mu, latents_logvar = trainer.extract_latents(state, pairs, grid_shapes)
        
        if latents_logvar is not None:
            # KL損失計算
            key = random.PRNGKey(42)
            latents, prior_kl, _ = trainer.lpn._sample_latents(latents_mu, latents_logvar, key)
            pairwise_kl = trainer.lpn._compute_pairwise_gaussian_kl(latents_mu, latents_logvar).mean()
            
            total_prior_kl += prior_kl * pairs.shape[0]
            total_pairwise_kl += pairwise_kl * pairs.shape[0]
        
        # ノルム計算
        batch_norms = jnp.linalg.norm(latents_mu, axis=-1).flatten()
        latent_norms.extend(batch_norms)
        
        total_samples += pairs.shape[0]
    
    # 平均計算
    avg_prior_kl = total_prior_kl / total_samples if total_prior_kl > 0 else 0
    avg_pairwise_kl = total_pairwise_kl / total_samples if total_pairwise_kl > 0 else 0
    avg_norm = np.mean(latent_norms)
    std_norm = np.std(latent_norms)
    
    print("\n📊 エンコーダー評価結果:")
    print(f"  事前分布KL: {avg_prior_kl:.6f}")
    print(f"  ペアワイズKL: {avg_pairwise_kl:.6f}")
    print(f"  潜在変数ノルム: {avg_norm:.4f} ± {std_norm:.4f}")
    
    return {
        "prior_kl": avg_prior_kl,
        "pairwise_kl": avg_pairwise_kl,
        "latent_norm_mean": avg_norm,
        "latent_norm_std": std_norm
    }

# 評価実行
evaluation_results = evaluate_encoder_quality(trained_state, val_dataloader)

## 11. 学習済みモデルの保存と読み込み

In [ ]:
import pickle
from flax.serialization import to_bytes, from_bytes

# モデル保存
def save_encoder_model(state, config, filepath="encoder_only_model.pkl"):
    """学習済みエンコーダーモデルを保存"""
    save_data = {
        "state": to_bytes(state),
        "encoder_config": config,
        "evaluation_results": evaluation_results,
        "training_history": trainer.train_metrics_history
    }
    
    with open(filepath, "wb") as f:
        pickle.dump(save_data, f)
    
    print(f"✅ モデルを保存しました: {filepath}")

# モデル読み込み
def load_encoder_model(filepath="encoder_only_model.pkl"):
    """保存されたエンコーダーモデルを読み込み"""
    with open(filepath, "rb") as f:
        save_data = pickle.load(f)
    
    # 新しいモデル作成
    dummy_trainer = EncoderOnlyTrainer(
        save_data["encoder_config"],
        decoder_config  # デコーダー設定は固定
    )
    
    # サンプルデータで初期化
    sample_batch = next(iter(val_dataloader))
    key = random.PRNGKey(42)
    dummy_state = dummy_trainer.init_train_state(key, sample_batch)
    
    # 保存された重みを復元
    restored_state = from_bytes(dummy_state, save_data["state"])
    
    print(f"✅ モデルを読み込みました: {filepath}")
    return restored_state, save_data

# モデル保存実行
save_encoder_model(trained_state, encoder_config)

# 保存したモデルの読み込みテスト
loaded_state, loaded_data = load_encoder_model()
print(f"読み込み成功: 学習ステップ数 = {loaded_state.step}")

## 12. まとめと次のステップ

In [ ]:
# 最終まとめ
print("🎉 エンコーダーのみ学習完了")
print("="*60)
print("\n📈 学習結果サマリー:")
print(f"  最終損失: {trainer.train_metrics_history[-1]['total_loss']:.6f}")
print(f"  最終事前分布KL: {trainer.train_metrics_history[-1]['prior_kl']:.6f}")
print(f"  最終ペアワイズKL: {trainer.train_metrics_history[-1]['pairwise_kl']:.6f}")
print(f"  最終潜在変数ノルム: {trainer.train_metrics_history[-1]['latents_norm']:.4f}")

print("\n🔍 学習された潜在空間の特徴:")
final_latents_mu, _ = extract_all_latents(trained_state, val_dataloader)
final_latents_flat = final_latents_mu.reshape(-1, final_latents_mu.shape[-1])
print(f"  平均絶対値: {np.mean(np.abs(final_latents_flat)):.4f}")
print(f"  次元間相関の最大値: {np.max(np.abs(np.corrcoef(final_latents_flat.T))- np.eye(final_latents_flat.shape[1])):.4f}")
print(f"  有効次元数（分散>0.01）: {np.sum(np.var(final_latents_flat, axis=0) > 0.01)}")

print("\n🚀 次のステップの提案:")
print("  1. 学習済みエンコーダーを使ったデコーダー学習")
print("  2. 潜在空間でのクラスタリング分析")
print("  3. 異なるアーキテクチャ（層数、次元数）での実験")
print("  4. 実際のARCタスクデータでの検証")
print("  5. 潜在変数の意味的解釈分析")

print("\n💾 保存されたファイル:")
print("  - encoder_only_model.pkl: 学習済みモデルと設定")
print("  - このノートブック: 再現可能な実験記録")